# qaari-eval — Kaggle dataset indexer (T4 GPU)

Runs the full v2 pipeline over **entire** ayah-by-ayah reciter datasets (EveryAyah) and produces
the 232-d fingerprints and benchmark rows. Work is sharded so each Kaggle session (≈ 9 h, 30 GPU-h
per week) processes a slice; re-running a shard resumes where it stopped.

Settings → Accelerator: **GPU T4**. Add a Kaggle secret `HF_TOKEN` to push results to the Hub.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/akadaan310/muqri.git qaari-eval
%cd qaari-eval
!pip install -q -r requirements-ml.txt nara_wpe

In [ ]:
# Which slice of the work this session does. 16 reciters x 6236 ayahs = ~100k ayah-analyses.
SHARD, N_SHARDS = 0, 8          # change SHARD per session (0..7)
VERSES = "all"                  # "all" = entire Qur'an, or "strategic" (59 ayahs) for a quick run
RECITERS = ""                   # e.g. "Husary_128kbps Yasser_Ad-Dussary_128kbps"; empty = full roster
MODES = "studio taraweeh_adapted"

In [ ]:
import os
import subprocess

cmd = f"python benchmarks/run_benchmark.py --verses {VERSES} --shard {SHARD}/{N_SHARDS} " \
      f"--modes {MODES} --output /kaggle/working/runs_shard{SHARD}.jsonl " \
      f"--cache-dir /kaggle/working/everyayah"
if RECITERS:
    cmd += f" --reciters {RECITERS}"
print(cmd)
subprocess.run(cmd, shell=True, check=True)

In [ ]:
# Merge every shard you have downloaded into benchmarks/results/ and rebuild the indices.
!cp /kaggle/working/runs_shard*.jsonl benchmarks/results/ 2>/dev/null; ls -la benchmarks/results
!python benchmarks/summarize.py

In [ ]:
# Optional: publish results + indices to the Hugging Face Hub.
from huggingface_hub import HfApi

token = os.environ.get("HF_TOKEN")
if token:
    api = HfApi(token=token)
    repo = "your-username/qaari-eval-fingerprints"   # change me
    api.create_repo(repo, repo_type="dataset", exist_ok=True)
    api.upload_folder(folder_path="benchmarks/results", repo_id=repo, repo_type="dataset", path_in_repo="results")
    api.upload_folder(folder_path="index", repo_id=repo, repo_type="dataset", path_in_repo="index")
else:
    print("Set the HF_TOKEN secret to upload.")